In [1]:
!pip install transformers accelerate sentencepiece torch gradio --quiet


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gradio as gr

MODEL = "mistralai/Mistral-7B-Instruct-v0.2"  # PUBLIC, NO TOKEN REQUIRED

tokenizer = AutoTokenizer.from_pretrained(MODEL ,use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Decoder-only Chat Model Loaded!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Decoder-only Chat Model Loaded!


In [ ]:
# ======================================================
# FIXED FUNCTIONS (Replace your old generate_reply and respond)
# ======================================================

def generate_reply(history, user_message):
    """
    history: list of {"role": "user" or "assistant", "content": "..."}
    user_message: latest user input (string)
    """
    # Start system message
    messages = [{"role": "system", "content": "You are a friendly helpful chatbot."}]
    last_role = "system"

    # Merge consecutive roles to ensure alternating user/assistant pattern
    for msg in history:
        role = msg.get("role", "user")
        content = msg.get("content", "")

        if role == last_role:
            messages[-1]["content"] += "\n" + content
        else:
            messages.append({"role": role, "content": content})
            last_role = role

    # Add the new user message, merging if necessary
    if last_role == "user":
        messages[-1]["content"] += "\n" + user_message
    else:
        messages.append({"role": "user", "content": user_message})
        last_role = "user"

    # Convert messages to chat template tokens
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
    ).to(model.device)

    # Generate bot output
    output = model.generate(
        input_ids,
        max_new_tokens=256,
        temperature=0.8,
        top_p=0.95,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    reply = decoded

    # Extract only assistant reply
    if "assistant" in decoded:
        reply = decoded.rsplit("assistant", 1)[-1].replace(":", "").strip()

    # Fallback: strip prompt if leaking
    flat_prompt = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    if reply.startswith(flat_prompt):
        reply = reply[len(flat_prompt):].strip()

    return reply


def build_html_from_history(history):
    """Builds the chat bubble HTML from the stored message history."""
    html = ""
    for msg in history:
        if msg["role"] == "user":
            html += f"<div class='user-msg'>{msg['content']}</div>"
        else:
            html += f"<div class='bot-msg'>{msg['content']}</div>"
    return html


def respond(user_message, history):
    """Handles sending user messages and generating bot responses."""
    if not user_message:
        return build_html_from_history(history), history

    # Append user bubble
    history.append({"role": "user", "content": user_message})

    # Generate bot reply
    bot_reply = generate_reply(history, user_message)

    # Append assistant bubble
    history.append({"role": "assistant", "content": bot_reply})

    # Build HTML chat UI
    html = build_html_from_history(history)
    return html, history


# ======================================================
# MESSENGER-STYLE GRADIO UI
# ======================================================

messenger_css = """
#chatbox {
    height: 500px;
    overflow-y: auto;
    border: 2px solid #ccc;
    padding: 10px;
    background: #f5f5f5;
}
.user-msg {
    background: #DCF8C6;
    padding: 10px;
    border-radius: 10px;
    width: fit-content;
    max-width: 80%;
    margin-bottom: 10px;
    margin-left: auto;
    font-size: 16px;
}
.bot-msg {
    background: #FFF;
    padding: 10px;
    border-radius: 10px;
    width: fit-content;
    max-width: 80%;
    margin-bottom: 10px;
    margin-right: auto;
    border: 1px solid #ddd;
    font-size: 16px;
}
"""

with gr.Blocks(css=messenger_css) as demo:
    gr.Markdown("## 💬 Messenger-Style Chatbot (Mistral — Decoder Only, No Login)")

    chatbox = gr.HTML("<div id='chatbox'></div>")
    user_input = gr.Textbox(label="Type your message here...")
    history = gr.State([])

    user_input.submit(respond, [user_input, history], [chatbox, history])
    user_input.submit(lambda: "", None, user_input)  # Clears text box after send

demo.launch(share=True, debug=True)


/tmp/ipython-input-1847689653.py:128: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=messenger_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://423edc6c82bab09497.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
